# Elementary potential flows, and superposition

Four building blocks. Laplace's equation is linear, so **you can just add them up** — that is
the whole trick of classical aerodynamics.

For an element at $(x_0,y_0)$, with $r,\theta$ measured from it:

| element | $\phi$ (velocity potential) | $\psi$ (stream function) |
|---|---|---|
| uniform stream $U$ at angle $\alpha$ | $U(x\cos\alpha + y\sin\alpha)$ | $U(y\cos\alpha - x\sin\alpha)$ |
| source, strength $\Lambda$ ($\Lambda<0$: sink) | $\dfrac{\Lambda}{2\pi}\ln r$ | $\dfrac{\Lambda}{2\pi}\theta$ |
| vortex, strength $\Gamma$ (CCW $+$) | $\dfrac{\Gamma}{2\pi}\theta$ | $-\dfrac{\Gamma}{2\pi}\ln r$ |
| doublet, strength $\kappa$ ($+x$ axis) | $\dfrac{\kappa}{2\pi}\dfrac{\cos\theta}{r}$ | $-\dfrac{\kappa}{2\pi}\dfrac{\sin\theta}{r}$ |

with $u = \partial\phi/\partial x = \partial\psi/\partial y$ and
$v = \partial\phi/\partial y = -\partial\psi/\partial x$.

Note the swap: the source and the vortex are the *same solution with $\phi$ and $\psi$
exchanged*. That is why a vortex has closed circular streamlines and a source has radial ones.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

STRENGTH = 2*np.pi        # default strength; with U=1 this puts features at r ~ 1
U_INF, ALPHA = 1.0, 0.0   # uniform stream speed and angle (radians)

def field(elements, X, Y, uniform=True):
    """Add up u, v, psi, phi for a list of elements.  Superposition, literally a sum."""
    u = np.zeros_like(X); v = np.zeros_like(X)
    psi = np.zeros_like(X); phi = np.zeros_like(X)

    if uniform:
        ca, sa = np.cos(ALPHA), np.sin(ALPHA)
        u += U_INF*ca; v += U_INF*sa
        psi += U_INF*(Y*ca - X*sa); phi += U_INF*(X*ca + Y*sa)

    for e in elements:
        dx, dy = X - e['x'], Y - e['y']
        r2 = np.maximum(dx*dx + dy*dy, 1e-9)
        th = np.arctan2(dy, dx)
        k = e['S']/(2*np.pi)

        if e['kind'] == 'source':
            u += k*dx/r2;  v += k*dy/r2
            psi += k*th;   phi += 0.5*k*np.log(r2)
        elif e['kind'] == 'vortex':
            u += -k*dy/r2; v += k*dx/r2
            psi += -0.5*k*np.log(r2); phi += k*th
        elif e['kind'] == 'doublet':
            u += k*(dy*dy - dx*dx)/r2**2; v += -k*2*dx*dy/r2**2
            psi += -k*dy/r2;              phi += k*dx/r2
    return u, v, psi, phi

## The widget

Click in **any** of the three panels to drop an element there. All three views update together:
the stream function, the velocity potential, and the velocity field itself.

| key | element |
|---|---|
| **1** | source (negative strength = sink) |
| **2** | vortex |
| **3** | doublet |

* **left-click** places the selected element, **right-click** removes the nearest one
* each element is numbered; its **slider below the plot** sets its strength (in units of $2\pi$).
  Negative flips it: a clockwise vortex, a doublet pointing along $-x$
* **`u`** toggles the uniform stream on/off, **`z`** undoes, **`d`** deletes everything
* click the figure once first so it has keyboard focus

Watch $\psi$ and $\phi$: their contours always cross at right angles, because
$\nabla\phi\cdot\nabla\psi = uv - vu = 0$ wherever the flow is not stagnant.

In [ ]:
%matplotlib widget
import ipywidgets as widgets

LIM, N = 3.0, 220
gx = np.linspace(-LIM, LIM, N)
X, Y = np.meshgrid(gx, gx)

elements = []
state = {'kind': 'source', 'uniform': True}
STYLE = {'source':  dict(marker='+', color='#d62728', ms=13, mew=2.5),
         'sink':    dict(marker='x', color='#1f77b4', ms=11, mew=2.5),   # a source with S < 0
         'vortex':  dict(marker='o', color='#2ca02c', ms=10, mew=2.5, mfc='none'),
         'doublet': dict(marker='*', color='#9467bd', ms=15)}

with plt.ioff():
    fig, axes = plt.subplots(1, 3, figsize=(14.5, 5.0))
sliders = widgets.VBox()

def levels(F, kind, n=24, most=60):
    """Evenly spaced contour levels.  Percentiles ignore the blow-up at the singularities.
    If elements of `kind` make F jump across a cut, the spacing is (strongest one's |S|)/n,
    offset by half a step: that element gets n even rays, none sits on its cut, and lines
    match up across it (weaker elements get proportionally fewer rays).  Too many levels (a weak source in a strong stream) -> coarsen by a
    divisor of n, which keeps both properties."""
    lo, hi = np.percentile(F, 2), np.percentile(F, 98)
    S = [abs(e['S']) for e in elements if e['kind'] == kind and e['S'] != 0]
    if not S:
        return np.linspace(lo, hi, n) if hi > lo else n
    d = max(S)/n
    for k in (1, 2, 3, 4, 6, 8, 12, 24):
        if (hi - lo)/(k*d) <= most:
            break
    d *= k
    return d*(np.arange(np.floor(lo/d) - 1, np.ceil(hi/d) + 1) + 0.5)

def mask_cuts(F, kinds):
    """Blank the grid rows either side of each element's theta = +-pi cut, so contour
    does not stack every level onto that one fake jump."""
    cut = np.zeros_like(F, dtype=bool)
    for e in elements:
        if e['kind'] in kinds:
            cut |= (X < e['x']) & (np.abs(Y - e['y']) < gx[1] - gx[0])
    return np.ma.masked_where(cut, F)

def redraw():
    u, v, psi, phi = field(elements, X, Y, uniform=state['uniform'])
    for ax in axes:
        ax.clear()

    axes[0].contour(X, Y, mask_cuts(psi, ('source',)), levels=levels(psi, 'source'),
                    colors='#1f77b4', linewidths=0.9, linestyles='solid')
    axes[0].set_title(r'stream function $\psi$   (= streamlines)')

    axes[1].contour(X, Y, mask_cuts(phi, ('vortex',)), levels=levels(phi, 'vortex'),
                    colors='#d62728', linewidths=0.9, linestyles='solid')
    axes[1].set_title(r'velocity potential $\phi$   (= equipotentials)')

    speed = np.hypot(u, v)
    # cap the scale near the singularities, or they wash out the whole picture
    vmax = 3*U_INF if state['uniform'] else max(np.percentile(speed, 90), 1e-6)
    axes[2].pcolormesh(X, Y, np.clip(speed, 0, vmax), cmap='viridis',
                       shading='auto', vmin=0, vmax=vmax)
    axes[2].streamplot(gx, gx, u, v, color='w', linewidth=0.7, density=1.1, arrowsize=0.8)
    axes[2].set_title('velocity field  (colour = speed)')

    for ax in axes:
        for i, e in enumerate(elements):
            style = STYLE['sink' if e['kind'] == 'source' and e['S'] < 0 else e['kind']]
            ax.plot([e['x']], [e['y']], ls='none', **style)
            ax.annotate(str(i + 1), (e['x'], e['y']), xytext=(7, 7), weight='bold',
                        textcoords='offset points', fontsize=8, color=style['color'])
        ax.set(xlim=(-LIM, LIM), ylim=(-LIM, LIM), aspect='equal')

    net_src = sum(e['S'] for e in elements if e['kind'] == 'source')
    circ = sum(e['S'] for e in elements if e['kind'] == 'vortex')
    fig.suptitle(f"tool = {state['kind'].upper()}  [1 source · 2 vortex · 3 doublet]"
                 f"   |   uniform stream {'ON' if state['uniform'] else 'OFF'}\n"
                 f"{len(elements)} elements,  net source = {net_src:+.2f},"
                 f"  total circulation = {circ:+.2f}   (lift = rho*U*Gamma)")
    fig.canvas.draw_idle()

def make_sliders():
    """One strength slider per element, numbered to match the plot."""
    def slider(i, e):
        w = widgets.FloatSlider(value=e['S']/(2*np.pi), min=-3, max=3, step=0.05,
                                description=f"{i + 1}  {e['kind']}", continuous_update=False,
                                style={'description_width': '90px'},
                                layout=widgets.Layout(width='450px'))
        w.observe(lambda ch: (e.update(S=2*np.pi*ch['new']), redraw()), names='value')
        return w
    sliders.children = [slider(i, e) for i, e in enumerate(elements)]

def on_click(event):
    if event.inaxes not in list(axes) or event.xdata is None:
        return
    if event.button == 3 and elements:                 # remove the nearest element
        d = [np.hypot(e['x'] - event.xdata, e['y'] - event.ydata) for e in elements]
        if min(d) < 0.6:
            elements.pop(int(np.argmin(d)))
    elif event.button == 1:
        elements.append(dict(kind=state['kind'], x=event.xdata, y=event.ydata, S=STRENGTH))
    make_sliders()
    redraw()

def on_key(event):
    keys = {'1': 'source', '2': 'vortex', '3': 'doublet'}
    if event.key in keys:
        state['kind'] = keys[event.key]
    elif event.key == 'u':
        state['uniform'] = not state['uniform']
    elif event.key == 'z' and elements:
        elements.pop()
    elif event.key == 'd':
        elements.clear()
    make_sliders()
    redraw()

fig.canvas.mpl_connect('button_press_event', on_click)
fig.canvas.mpl_connect('key_press_event', on_key)
redraw()
fig.tight_layout()
widgets.VBox([fig.canvas, sliders])

: 

## Things to build

With the uniform stream on, and everything placed at the origin unless stated:

| build this | how | what to look for |
|---|---|---|
| **Rankine half-body** | one **source** | stagnation point at $x=-\Lambda/2\pi U = -1$; the body is open at the back |
| **Rankine oval** | **source** at $x=-1$, another at $x=+1$ with its slider at $-1$ | a closed body; net source $=0$ is exactly why it closes |
| **Cylinder** | one **doublet** | radius $a=\sqrt{\kappa/2\pi U}=1$; $u=2U$ at the top |
| **Lifting cylinder** | **doublet** + **vortex** | stagnation points slide round; the streamline pattern loses its up–down symmetry, and that asymmetry *is* the lift |
| **Nothing at all** | press **u** to kill the uniform stream, then one source | pure radial flow |

Two things worth pointing out to a class:

* **Net source strength decides whether the body closes.** Source-only is a half-body that never
  shuts; a source $+$ an equal sink gives a closed oval. A real aerofoil must have zero net source.
* **$\psi$ and $\phi$ have a branch cut, and it is not physics.** Both
  $\psi_{\rm source} = (\Lambda/2\pi)\theta$ and $\phi_{\rm vortex} = (\Gamma/2\pi)\theta$ are
  *multivalued*: go once round the element and they jump by $\Lambda$ and $\Gamma$. Left alone,
  the contour routine piles every level onto that $\theta = \pm\pi$ branch cut (a thick fake ray
  to the left of the element), so `mask_cuts` blanks those grid rows. The velocity panel needs
  no such fix: $u$ and $v$ are perfectly single-valued.

  That jump is not a nuisance to be hidden, it is the point: the jump in $\phi$ round a vortex
  *is* $\oint\mathbf{V}\cdot d\mathbf{l} = \Gamma$ from Tutorial 2. A single-valued $\phi$ would
  mean zero circulation, hence zero lift.

* The speed panel's colour scale is capped at $3U_\infty$; anything saturated is inside a body
  or right on top of a singularity, where the potential-flow solution does not describe real
  fluid anyway.